### Just an example if TypeSafe's : __Jev__

In [3]:
from enum import Enum
from typing import Annotated, Literal

from pydantic import BaseModel, ConfigDict

from pydantic_ai import Agent, BoolCriteria, UseEnumMemberDocstrings


class Area(UseEnumMemberDocstrings, str, Enum):
    billing = 'billing'
    """Charges, invoices, plans and payment methods."""

    bug = 'bug'
    """Part of the product does not work as it should."""

    account = 'account'
    """Logging in, access, and account settings."""


class Ticket(BaseModel):
    """Triage a support ticket."""

    model_config = ConfigDict(use_attribute_docstrings=True)

    area: Area
    """Which team owns this ticket?"""

    urgent: Annotated[
        bool,
        BoolCriteria(
            true='The customer is losing money or has a deadline today.',
            false='It can wait its turn in the queue.',
        ),
    ]
    """Should this ticket jump the queue?"""

    app: Literal['web', 'ios', 'android'] | None
    """Which app is it about?"""


agent = Agent('typesafe:jev-latest', output_type=Ticket)
result = agent.run_sync(
    'The timeline on my Android phone has been blank since the update this morning, '
    'and my standup is in ten minutes.'
)
print(result.output)
#> area=<Area.bug: 'bug'> urgent=True app='android'
assert result.response.provider_details is not None
print(result.response.provider_details['confidence'])
#> {'area': 1.0, 'urgent': 0.24, 'app': 0.81}

UserError: Set the `TYPESAFE_API_KEY` environment variable or pass it via `TypeSafeProvider(api_key=...)` to use the TypeSafe provider. To try Pydantic AI without an API key, use the built-in test model: `Agent('test')`. See https://pydantic.dev/docs/ai/guides/testing/